# Тема 4. Инструменты агента и механизм их вызова (tool calling)

**Модуль II · Пример 1 из 4**

Сквозной пример модуля — ассистент службы поддержки интернет-магазина **«ШопБот»**. В этой тетради мы разбираем **инструменты (tools)** — как языковая модель вызывает внешние функции, и как спроектировать инструмент в виде **надёжного контракта** между моделью и информационной системой.

### Что вы увидите
1. Подключение к языковой модели через OpenAI-совместимый API.
2. Инструмент как обычная Python-функция с типизированными входом и выходом.
3. **Ручной** разбор вызова инструмента (stop-and-parse) — чтобы понять механику «изнутри».
4. Проектирование надёжного инструмента: структурированные ошибки для модели, идемпотентность, повторные вызовы, частичные отказы.
5. **Библиотечная абстракция** — генерация JSON-схемы инструмента из Pydantic-модели и цикл агента.

### Используемые библиотеки
- **openai** — официальный клиент; работает с любым OpenAI-совместимым API.
- **pandas** — загрузка каталога товаров из CSV.
- **pydantic** — описание и валидация типизированных аргументов инструмента.

## Шаг 0. Конфигурация доступа к модели

Мы обращаемся к моделям через **OpenAI-совместимый API**. Достаточно указать `base_url` и `api_key` — остальной код идентичен работе с OpenAI.

- `qwen/qwen3.7-max` — основная модель, уверенно поддерживает вызов инструментов.
- `openai/gpt-oss-120b` — рассуждающая (reasoning) модель; ей нужен запас `max_tokens`, т.к. часть токенов уходит на скрытые рассуждения.

In [1]:
import os
from openai import OpenAI  # официальный клиент OpenAI; работает с любым OpenAI-совместимым API

# Ключ и адрес OpenAI-совместимого эндпоинта.
# В реальном проекте храните ключ в переменных окружения / секрет-хранилище.
API_KEY = os.environ.get("LLM_API_KEY", "YOUR_API_KEY")
BASE_URL = os.environ.get("LLM_BASE_URL", "YOUR_API_BASE_URL")

MODEL = "qwen/qwen3.7-max"          # модель с надёжным tool calling
REASONING_MODEL = "openai/gpt-oss-120b"

# OpenAI(...) создаёт клиента; base_url задаёт адрес эндпоинта.
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

# Проверим связь одним коротким запросом.
ping = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Ответь одним словом: готов"}],
    max_tokens=20,
)
print("Модель ответила:", ping.choices[0].message.content)

Модель ответила: Готов


## Шаг 1. Данные предметной области — каталог товаров

Инструменту нужен «реальный мир», к которому он обращается. Возьмём каталог из 500 товаров, построенный из открытого датасета **UCI Online Retail II** (транзакции интернет-магазина). Колонки: `sku` (артикул), `name` (название), `price` (цена), `sold` (сколько продано — прокси популярности).

`pandas.read_csv` читает CSV в объект `DataFrame` — таблицу с колонками, по которой удобно фильтровать и искать.

In [2]:
import pandas as pd

CATALOG = pd.read_csv("data/products.csv")
print("Товаров в каталоге:", len(CATALOG))
CATALOG.head(5)

Товаров в каталоге: 500


,sku,name,price,sold
0,85123A,White Hanging Heart T-Light Holder,2.95,58487
1,84077,World War 2 Gliders Asstd Designs,0.21,55091
2,17003,Brocade Ring Purse,0.21,48374
3,21212,Pack Of 72 Retro Spot Cake Cases,0.55,46755
4,84879,Assorted Colour Bird Ornament,1.69,45348


## Шаг 2. Инструмент как типизированная функция

Инструмент по определению курса состоит из трёх частей:
1. **текстовое описание** назначения (его читает модель, чтобы решить, когда вызывать);
2. **вызываемая программная сущность** (Python-функция);
3. **типизированные вход и выход**.

Начнём с простой функции поиска товаров. Обратите внимание: она возвращает **структуру** (список словарей), а не свободный текст — это упрощает и надёжную передачу результата модели, и последующую обработку.

In [3]:
def search_products(query: str, max_price: float | None = None, limit: int = 5) -> list[dict]:
    """Поиск товаров в каталоге по подстроке названия.

    query     — искомое слово/фраза (регистр не важен);
    max_price — верхний порог цены (необязательно);
    limit     — сколько позиций вернуть.
    Возвращает список словарей {sku, name, price}.
    """
    df = CATALOG[CATALOG["name"].str.contains(query, case=False, na=False)]
    if max_price is not None:
        df = df[df["price"] <= max_price]
    df = df.sort_values("sold", ascending=False).head(limit)
    return df[["sku", "name", "price"]].to_dict("records")

# Проверяем функцию напрямую, без модели:
search_products("bag", max_price=5.0, limit=3)

[{'sku': '85099B', 'name': 'Jumbo Bag Red Retrospot', 'price': 1.95},
 {'sku': '85099F', 'name': 'Jumbo Bag Strawberry', 'price': 1.95},
 {'sku': '20725', 'name': 'Lunch Bag Red Spotty', 'price': 1.65}]

## Шаг 3. Описание инструмента для модели (JSON Schema)

Чтобы модель знала о функции, инструмент описывают в формате **JSON Schema** — имя, назначение, параметры и их типы. Модель не исполняет код: она лишь **генерирует намерение вызвать** функцию с конкретными аргументами. Исполняем мы сами.

Ключевые поля:
- `name` — должно совпадать с тем, что мы потом вызовем;
- `description` — по нему модель решает, уместен ли вызов; пишите его для модели, как инструкцию;
- `parameters` — JSON-схема аргументов; `required` перечисляет обязательные.

In [4]:
search_tool_schema = {
    "type": "function",
    "function": {
        "name": "search_products",
        "description": "Найти товары в каталоге магазина по ключевому слову в названии. "
                       "Используй, когда пользователь ищет или подбирает товар.",
        "parameters": {
            "type": "object",
            "properties": {
                "query":     {"type": "string", "description": "Ключевое слово или фраза"},
                "max_price": {"type": "number", "description": "Максимальная цена, необязательно"},
                "limit":     {"type": "integer", "description": "Сколько позиций вернуть (по умолчанию 5)"},
            },
            "required": ["query"],
        },
    },
}
print("Схема инструмента описана.")

Схема инструмента описана.


## Шаг 4. Ручной разбор вызова (stop-and-parse)

Разберём механику по шагам, как она устроена «изнутри»:

1. Отправляем модели сообщение пользователя **вместе со списком инструментов**.
2. Модель либо отвечает текстом, либо возвращает **`tool_calls`** — имя функции и аргументы (в виде JSON-строки). На этом шаге генерация останавливается — это и есть **stop-and-parse**: без остановки модель начала бы *выдумывать* результат инструмента.
3. Мы разбираем аргументы, **исполняем** настоящую функцию и кладём результат обратно в диалог сообщением с ролью `tool`.
4. Снова зовём модель — теперь она формулирует ответ пользователю на основе результата.

Ниже — один полный цикл вручную.

In [5]:
import json

# Реестр: имя инструмента -> реальная функция. По нему исполняем то, что выбрала модель.
TOOL_FUNCS = {"search_products": search_products}

user_msg = "Посоветуй недорогую сумку, до 5 фунтов"
messages = [
    {"role": "system", "content":
        "Ты — ассистент интернет-магазина. Отвечай кратко и по делу. "
        "Каталог товаров — на английском языке, поэтому в аргумент query "
        "передавай английские ключевые слова (например, «сумка» -> «bag»)."},
    {"role": "user", "content": user_msg},
]

# --- Раунд 1: модель решает, нужен ли инструмент ---
resp = client.chat.completions.create(
    model=MODEL, messages=messages, tools=[search_tool_schema], max_tokens=500,
)
choice = resp.choices[0].message
print("Модель запросила вызовов инструментов:", len(choice.tool_calls or []))

# Кладём ответ модели (с намерением вызова) в историю
messages.append(choice.model_dump())

# --- Разбор и исполнение каждого вызова ---
for tc in (choice.tool_calls or []):
    fn_name = tc.function.name
    fn_args = json.loads(tc.function.arguments)   # аргументы приходят строкой JSON
    print(f"  -> {fn_name}({fn_args})")
    result = TOOL_FUNCS[fn_name](**fn_args)       # исполняем НАСТОЯЩУЮ функцию
    # Результат возвращаем сообщением с ролью tool и тем же tool_call_id
    messages.append({
        "role": "tool",
        "tool_call_id": tc.id,
        "content": json.dumps(result, ensure_ascii=False),
    })

# --- Раунд 2: модель формулирует ответ пользователю по результату инструмента ---
final = client.chat.completions.create(model=MODEL, messages=messages, max_tokens=500)
print("\nОтвет ассистента:\n", final.choices[0].message.content)

Модель запросила вызовов инструментов: 1
  -> search_products({'query': 'bag', 'max_price': 5, 'limit': 5})



Ответ ассистента:
 Вот несколько отличных вариантов сумок до 5 фунтов:

1. **Lunch Bag Red Spotty** — £1.65
   Компактная сумка для ланча в красный горошек.

2. **Jumbo Bag Red Retrospot** — £1.95
   Большая вместительная сумка в ретро-стиле (красный горошек).

3. **Jumbo Bag Strawberry** — £1.95
   Большая сумка с принтом клубники.

4. **Jumbo Bag Baroque Black White** — £1.95
   Большая сумка в чёрно-белом барочном стиле.

5. **Jumbo Storage Bag Suki** — £1.95
   Большая сумка для хранения с ярким дизайном.

Если нужна сумка для покупок — берите любой **Jumbo Bag**. Для обеда отлично подойдёт **Lunch Bag Red Spotty**. Что вас заинтересовало?


## Шаг 5. Надёжный инструмент: структурированные ошибки, идемпотентность, повторные вызовы

Тезис курса: **качественный инструмент — это надёжный контракт**, устойчивый к сбоям, повторным вызовам и изменениям среды. Разберём три свойства на инструменте оформления заказа.

### 5.1. Ошибки — для модели, а не только для разработчика
Если бросить голое исключение, модель увидит стектрейс и «растеряется». Правильно — вернуть **структурированное сообщение об ошибке**, из которого модель поймёт, что делать (переспросить пользователя, попробовать иначе).

### 5.2. Идемпотентность
Оформление заказа — **не**-идемпотентная операция: два вызова = два заказа. Защита — **ключ идемпотентности**: повторный вызов с тем же ключом возвращает тот же результат, а не создаёт дубль. Для сравнения, `search_products` идемпотентна: сколько ни зови — состояние мира не меняется.

In [6]:
import uuid

ORDERS = {}                 # хранилище заказов: order_id -> заказ
_IDEMPOTENCY = {}           # ключ идемпотентности -> order_id (защита от дублей)

def place_order(sku: str, quantity: int = 1, idempotency_key: str | None = None) -> dict:
    """Оформить заказ на товар. Не-идемпотентная операция, защищённая ключом идемпотентности."""
    # Повторный вызов с тем же ключом не создаёт новый заказ:
    if idempotency_key and idempotency_key in _IDEMPOTENCY:
        oid = _IDEMPOTENCY[idempotency_key]
        return {"status": "ok", "order_id": oid, "note": "повтор: возвращён существующий заказ",
                **ORDERS[oid]}

    # Валидация аргументов и понятная ошибка ДЛЯ МОДЕЛИ:
    row = CATALOG[CATALOG["sku"] == sku]
    if row.empty:
        return {"status": "error", "code": "sku_not_found",
                "message": f"Артикул {sku!r} не найден. Уточни SKU через search_products."}
    if quantity < 1:
        return {"status": "error", "code": "bad_quantity",
                "message": "Количество должно быть >= 1. Переспроси пользователя."}

    oid = "ORD-" + uuid.uuid4().hex[:8]
    order = {"order_id": oid, "sku": sku, "name": row.iloc[0]["name"],
             "price": float(row.iloc[0]["price"]), "quantity": int(quantity)}
    ORDERS[oid] = order
    if idempotency_key:
        _IDEMPOTENCY[idempotency_key] = oid
    return {"status": "ok", **order}

# Демонстрация идемпотентности: два вызова с одним ключом -> один заказ
k = "user42-cart-checkout-1"
r1 = place_order("85123A", 2, idempotency_key=k)
r2 = place_order("85123A", 2, idempotency_key=k)   # повтор (например, ретрай из-за таймаута сети)
print("Первый вызов :", r1["order_id"], r1.get("note", ""))
print("Повторный    :", r2["order_id"], r2.get("note", ""))
print("Заказов создано всего:", len(ORDERS), "(дубль не создан)")

# Демонстрация структурированной ошибки:
print("\nОшибка на несуществующем SKU:")
print(place_order("NOPE-000", 1))

Первый вызов : ORD-581cc116 
Повторный    : ORD-581cc116 повтор: возвращён существующий заказ
Заказов создано всего: 1 (дубль не создан)

Ошибка на несуществующем SKU:
{'status': 'error', 'code': 'sku_not_found', 'message': "Артикул 'NOPE-000' не найден. Уточни SKU через search_products."}


### 5.3. Частичный отказ и компенсирующая операция

Если действие состоит из нескольких шагов (списать со склада → создать заказ), а один из них падает, нужна **компенсирующая операция** — откат уже выполненного, чтобы система не осталась в противоречивом состоянии. Смоделируем резервирование склада с откатом.

In [7]:
STOCK = {"85123A": 3}   # остаток на складе по SKU

def reserve_and_order(sku: str, quantity: int) -> dict:
    """Двухшаговая операция: (1) резерв склада, (2) создание заказа.
    При отказе шага 2 выполняется компенсация шага 1 (возврат резерва)."""
    have = STOCK.get(sku, 0)
    if have < quantity:
        return {"status": "error", "code": "out_of_stock",
                "message": f"На складе только {have} шт. Предложи меньшее количество."}
    STOCK[sku] = have - quantity          # шаг 1: резерв
    try:
        order = place_order(sku, quantity)  # шаг 2: заказ
        if order["status"] != "ok":
            raise RuntimeError(order["message"])
        return order
    except Exception as e:
        STOCK[sku] = have                 # КОМПЕНСАЦИЯ: возвращаем резерв
        return {"status": "error", "code": "rolled_back",
                "message": f"Заказ не создан ({e}); резерв склада возвращён."}

print("Резерв 2 шт из 3:", reserve_and_order("85123A", 2)["status"], "| остаток:", STOCK["85123A"])
print("Резерв 5 шт (не хватит):", reserve_and_order("85123A", 5)["code"], "| остаток:", STOCK["85123A"])

Резерв 2 шт из 3: ok | остаток: 1
Резерв 5 шт (не хватит): out_of_stock | остаток: 1


## Шаг 6. Библиотечная абстракция: схема из Pydantic + декоратор

Писать JSON-схему руками — источник ошибок (описание разошлось с реальной сигнатурой). На практике схему **генерируют** из типов. Соберём мини-абстракцию:

- **pydantic.BaseModel** описывает аргументы с типами и ограничениями и умеет отдавать JSON-схему методом `model_json_schema()`;
- декоратор `@tool` связывает функцию, её описание и схему в один объект;
- функция `run_agent` реализует полный цикл (модель → инструменты → модель) в while-петле, поддерживая **несколько шагов** и несколько инструментов.

Так выглядит то, что «под капотом» у библиотек вроде LangChain, но в прозрачном учебном виде.

In [8]:
from pydantic import BaseModel, Field
from typing import Callable

class Tool:
    """Обёртка: функция + сгенерированная из Pydantic схема для модели."""
    def __init__(self, func: Callable, args_model: type[BaseModel], description: str):
        self.func, self.args_model, self.name = func, args_model, func.__name__
        self.description = description
    def schema(self) -> dict:
        params = self.args_model.model_json_schema()
        params.pop("title", None)
        return {"type": "function", "function": {
            "name": self.name, "description": self.description, "parameters": params}}
    def run(self, **kwargs):
        validated = self.args_model(**kwargs)      # Pydantic валидирует и приводит типы
        return self.func(**validated.model_dump())

def tool(args_model: type[BaseModel], description: str):
    def wrap(func): return Tool(func, args_model, description)
    return wrap

# --- Описываем аргументы моделями Pydantic (типы + подсказки) ---
class SearchArgs(BaseModel):
    query: str = Field(description="Ключевое слово в названии товара")
    max_price: float | None = Field(default=None, description="Максимальная цена")
    limit: int = Field(default=5, ge=1, le=20, description="Сколько позиций вернуть")

class OrderArgs(BaseModel):
    sku: str = Field(description="Артикул товара (SKU)")
    quantity: int = Field(default=1, ge=1, description="Количество")

@tool(SearchArgs, "Найти товары по ключевому слову в названии.")
def search_products_t(query, max_price=None, limit=5):
    return search_products(query, max_price, limit)

@tool(OrderArgs, "Оформить заказ на товар по его SKU.")
def place_order_t(sku, quantity=1):
    return place_order(sku, quantity, idempotency_key=f"agent-{sku}-{quantity}")

print("Схема, сгенерированная из Pydantic:")
print(json.dumps(search_products_t.schema(), ensure_ascii=False, indent=2))

Схема, сгенерированная из Pydantic:
{
  "type": "function",
  "function": {
    "name": "search_products_t",
    "description": "Найти товары по ключевому слову в названии.",
    "parameters": {
      "properties": {
        "query": {
          "description": "Ключевое слово в названии товара",
          "title": "Query",
          "type": "string"
        },
        "max_price": {
          "anyOf": [
            {
              "type": "number"
            },
            {
              "type": "null"
            }
          ],
          "default": null,
          "description": "Максимальная цена",
          "title": "Max Price"
        },
        "limit": {
          "default": 5,
          "description": "Сколько позиций вернуть",
          "maximum": 20,
          "minimum": 1,
          "title": "Limit",
          "type": "integer"
        }
      },
      "required": [
        "query"
      ],
      "type": "object"
    }
  }
}


## Шаг 7. Полный цикл агента с несколькими инструментами

`run_agent` крутит петлю «модель ⇄ инструменты», пока модель не перестанет запрашивать вызовы (или пока не исчерпается лимит шагов — простая защита от зацикливания, к которой мы вернёмся в теме про guardrails). Это уже полноценный агент: он сам выбирает, какие инструменты и в каком порядке вызывать.

In [9]:
def run_agent(user_message: str, tools: list[Tool], model: str = MODEL, max_steps: int = 5):
    registry = {t.name: t for t in tools}
    schemas = [t.schema() for t in tools]
    messages = [
        {"role": "system", "content":
            "Ты — ассистент интернет-магазина ШопБот. Помогай найти товар и оформить заказ. "
            "Всегда уточняй SKU через поиск, прежде чем оформлять заказ. "
            "Каталог — на английском; в query передавай английские ключевые слова "
            "(«держатель» -> «holder», «свеча» -> «candle»)."},
        {"role": "user", "content": user_message},
    ]
    for step in range(max_steps):
        resp = client.chat.completions.create(
            model=model, messages=messages, tools=schemas, max_tokens=800)
        msg = resp.choices[0].message
        messages.append(msg.model_dump())
        if not msg.tool_calls:                       # инструменты больше не нужны — готов ответ
            return msg.content, messages
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            print(f"[шаг {step+1}] вызов {tc.function.name}({args})")
            result = registry[tc.function.name].run(**args)
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(result, ensure_ascii=False)})
    return "Достигнут лимит шагов.", messages

answer, _ = run_agent(
    "Найди в каталоге товар White Hanging Heart T-Light Holder и оформи заказ на 1 штуку",
    tools=[search_products_t, place_order_t],
)
print("\n=== Ответ агента ===\n", answer)

[шаг 1] вызов search_products_t({'query': 'White Hanging Heart T-Light Holder', 'limit': 5})


[шаг 2] вызов place_order_t({'sku': '85123A', 'quantity': 1})



=== Ответ агента ===
 Ваш заказ успешно оформлен! 🎉

Детали:
- Номер заказа: **ORD-13247138**
- Товар: White Hanging Heart T-Light Holder
- Количество: 1 шт.
- Цена: £2.95


## Итоги

- Инструмент — это **описание для модели + реальная функция + типизированный контракт**. Модель лишь *выбирает* вызов; исполняем мы.
- **stop-and-parse**: без остановки на вызове модель выдумала бы результат. Возврат в диалог идёт сообщением роли `tool`.
- Надёжный инструмент возвращает **структурированные ошибки для модели**, **идемпотентен** (или защищён ключом идемпотентности), а многошаговые действия имеют **компенсацию** при частичном отказе.
- Схему инструмента лучше **генерировать из типов** (Pydantic), а не писать руками, — так описание не расходится с реализацией.

**Дальше:** в Теме 5 те же инструменты мы вынесем в отдельный **MCP-сервер**, чтобы отвязать их от конкретного агента.